#     # Q3: Can a simple regression model beat a naive forecast?

In [1]:
#libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Visual configurations
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

df_final = pd.read_parquet("../data/processed/pld_se_co_monthly.parquet")

# 1. Target at month t, predictors from month t-1 (information available before the month starts)
df_q3 = pd.DataFrame(index=df_final.index)
df_q3["pld"] = df_final["pld_se_co"]
df_q3["pld_lag1"] = df_final["pld_se_co"].shift(1)
df_q3["ear_lag1"] = df_final["ear_seco_mwmes"].shift(1) / 1000
df_q3["ena_lag1"] = df_final["ena_seco_mwmed"].shift(1) / 1000  

# 2. The first month has no previous month, so it is dropped
df_q3 = df_q3.dropna()

features_1 = ["ear_lag1", "ena_lag1"]
features_2 = ["pld_lag1", "ear_lag1", "ena_lag1"]

print(f"Months available: {len(df_q3)}")
display(df_q3.head())

Months available: 104


,pld,pld_lag1,ear_lag1,ena_lag1
date,,,,
2017-02-01,128.43,121.44,69.836613,43.990903
2017-03-01,216.24,128.43,80.383464,49.413429
2017-04-01,371.47,216.24,83.858097,45.626032
2017-05-01,411.49,371.47,84.446433,39.045967
2017-06-01,124.70,411.49,85.763161,38.930387


* The dataset has 104 months: the original 105 minus the first month (January 2017), which has no previous month to provide the lagged values.
* The lag works as intended. Each `pld_lag1` equals the `pld` of the row above (for example, the March PLD of 216.24 appears as the lag of April), and the first row (February 2017) uses the EAR and ENA of January 2017 (EAR of ~69.8 thousand MWmês and ENA of ~44.0 thousand MWmed).
* EAR and ENA are expressed in thousands of MWmês and MWmed, the same scale used in the regression of Q2.

In [ ]:
# --- Q3: WALK-FORWARD FORECAST ---

# Train on 2017-2020 first, then forecast 2021 onward, one month at a time
first_test_month = "2021-01-01"
test_dates = df_q3.loc[first_test_month:].index

results = []

for date in test_dates:
    # 1. Training data: only months BEFORE the month being forecast
    train = df_q3.loc[df_q3.index < date]
    test_row = df_q3.loc[[date]]

    # 2. Floor protection: the forecast cannot go below the lowest PLD seen so far
    floor = train["pld"].min()

    # 3. Naive forecast: this month's PLD = previous month's PLD
    naive_pred = test_row["pld_lag1"].iloc[0]

    # 4. Model 1: EAR and ENA of the previous 
    model_1 = LinearRegression()
    model_1.fit(train[features_1], train["pld"])
    pred_1 = model_1.predict(test_row[features_1])[0]
    if pred_1 < floor:
        pred_1 = floor

    # 5. Model 2: previous PLD + EAR + ENA
    model_2 = LinearRegression()
    model_2.fit(train[features_2], train["pld"])
    pred_2 = model_2.predict(test_row[features_2])[0]
    if pred_2 < floor:
        pred_2 = floor

    results.append({
        "date": date,
        "actual": test_row["pld"].iloc[0],
        "naive": naive_pred,
        "model_1": pred_1,
        "model_2": pred_2,
    })

df_results = pd.DataFrame(results).set_index("date")
print(f"Forecasted months: {len(df_results)}")
display(df_results.head())

Forecasted months: 57


,actual,naive,model_1,model_2
date,,,,
2021-01-01,242.72,267.49,319.266342,253.244042
2021-02-01,165.98,242.72,238.772507,207.953298
2021-03-01,109.02,165.98,201.277594,155.078736
2021-04-01,132.63,109.02,180.775126,128.582924
2021-05-01,218.70,132.63,249.038950,184.876503
